In [2]:
import anndata as ad
import pandas as pd
import numpy as np
import tables
import h5py
from scipy import sparse
from tqdm import tqdm
from pathlib import Path
import re

## Create gene list - intersection of all Ensembl ID sets from all datasets

In [2]:
archs4 = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/ARCHS4/archs4.h5ad", backed = "r")
archs4_genes = set(archs4.var_names)
del archs4

In [3]:
depmap = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/DepMap/depmap.h5ad", backed = "r")
depmap_genes = set(depmap.var_names)
del depmap

In [3]:
disignatlas = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/DiSignAtlas/disignatlas.h5ad", backed = "r")
disignatlas_genes = set(disignatlas.var_names)
del disignatlas

In [5]:
gdsc = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/GDSC/gdsc.h5ad", backed = "r")
gdsc_genes = set(gdsc.var_names)
del gdsc

In [6]:
gtex = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/GTEx/gtex.h5ad", backed = "r")
gtex_genes = set(gtex.var_names)
del gtex

In [7]:
lincs = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/LINCS/lincs.h5ad", backed = "r")
lincs_genes = set(lincs.var_names)
del lincs

In [17]:
survboard_data_dir = Path("/cluster/work/boeva/eheiss/datasets/SurvBoard/data_reproduced")
gene_info = pd.read_csv("/cluster/work/boeva/eheiss/scbFM/data/bulkformer_gene_info.csv")

sym2ensg = dict(zip(gene_info["gene_symbol"].astype(str), gene_info["ensg_id"].astype(str)))
sym2ensg_upper = {k.upper(): v for k, v in sym2ensg.items()}

def map_gex_col_to_ensg(col):
    raw = col[len("gex_"):] if col.startswith("gex_") else col
    parts = [p.strip() for p in raw.split("|") if p.strip()]

    # Already Ensembl ID, with or without version: ENSG00000123456.7 -> ENSG00000123456
    for p in parts + [raw]:
        m = re.match(r"^(ENSG\d+)(?:\.\d+)?$", p)
        if m:
            return m.group(1)

    # Standard TCGA SurvBoard columns: gex_HUGO|ENTREZ
    symbol = parts[0] if parts else raw
    candidates = [
        symbol,
        symbol.upper(),
        symbol.replace(".", "-"),
        symbol.replace(".", "-").upper(),
    ]

    for s in candidates:
        if s in sym2ensg:
            return sym2ensg[s]
        if s in sym2ensg_upper:
            return sym2ensg_upper[s]

    return None

# TCGA-only SurvBoard expression files
survboard_files = sorted(
    (survboard_data_dir / "TCGA").glob("*_data_complete_modalities_preprocessed.csv")
)

if not survboard_files:
    raise FileNotFoundError(f"No TCGA SurvBoard files found under {survboard_data_dir / 'TCGA'}")

survboard_gene_sets = {}
mapping_rows = []

for path in survboard_files:
    project = path.parent.name
    cancer = path.name.replace("_data_complete_modalities_preprocessed.csv", "")

    header = pd.read_csv(path, nrows=0)
    gex_cols = [c for c in header.columns if c.startswith("gex_")]

    mapped = []
    unmapped = []
    seen = set()

    for col in gex_cols:
        ensg = map_gex_col_to_ensg(col)
        if ensg is None:
            unmapped.append(col)
            continue
        if ensg not in seen:
            mapped.append(ensg)
            seen.add(ensg)

    gene_set = set(mapped)
    survboard_gene_sets[(project, cancer)] = gene_set

    mapping_rows.append({
        "project": project,
        "cancer": cancer,
        "n_gex_cols": len(gex_cols),
        "n_mapped_unique_ensg": len(gene_set),
        "n_unmapped_or_duplicate": len(gex_cols) - len(gene_set),
        "n_unmapped_symbols": len(set(unmapped)),
        "example_unmapped": unmapped[:5],
    })

survboard_mapping_summary = pd.DataFrame(mapping_rows).sort_values(["project", "cancer"])
display(survboard_mapping_summary)

survboard_genes = set().union(*survboard_gene_sets.values())
survboard_genes_intersection = set.intersection(*survboard_gene_sets.values())

print("SurvBoard project: TCGA only")
print("SurvBoard files:", len(survboard_files))
print("SurvBoard mapped ENSG union:", len(survboard_genes))
print("SurvBoard mapped ENSG intersection:", len(survboard_genes_intersection))


,project,cancer,n_gex_cols,n_mapped_unique_ensg,n_unmapped_or_duplicate,n_unmapped_symbols,example_unmapped
0,TCGA,BLCA,20531,16522,4009,4009,"[gex_?|100130426, gex_?|100133144, gex_?|10013..."
1,TCGA,BRCA,20531,16522,4009,4009,"[gex_?|100130426, gex_?|100133144, gex_?|10013..."
2,TCGA,CESC,20531,16522,4009,4009,"[gex_?|100130426, gex_?|100133144, gex_?|10013..."
3,TCGA,COAD,17507,14520,2987,2987,"[gex_?|100133144, gex_?|100134869, gex_?|10357..."
4,TCGA,ESCA,19076,15670,3406,3406,"[gex_?|100133144, gex_?|100134869, gex_?|10357..."
5,TCGA,GBM,20531,16522,4009,4009,"[gex_?|100130426, gex_?|100133144, gex_?|10013..."
6,TCGA,HNSC,20531,16522,4009,4009,"[gex_?|100130426, gex_?|100133144, gex_?|10013..."
7,TCGA,KIRC,20531,16522,4009,4009,"[gex_?|100130426, gex_?|100133144, gex_?|10013..."
8,TCGA,KIRP,20531,16522,4009,4009,"[gex_?|100130426, gex_?|100133144, gex_?|10013..."
9,TCGA,LAML,16765,13868,2897,2897,"[gex_?|100133144, gex_?|100134869, gex_?|10357..."


SurvBoard project: TCGA only
SurvBoard files: 21
SurvBoard mapped ENSG union: 16522
SurvBoard mapped ENSG intersection: 13575


In [9]:
tcga = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/TCGA/tcga.h5ad", backed = "r")
tcga_genes = set(tcga.var_names)
del tcga

In [18]:
print("ARCHS4 genes:", len(archs4_genes))
print("DepMap genes:", len(depmap_genes))
print("DisSignAtlas genes:", len(disignatlas_genes))
print("GDSC genes:", len(gdsc_genes))
print("GTEx genes:", len(gtex_genes))
print("LINCS genes:", len(lincs_genes))
print("TCGA genes:", len(tcga_genes))
print("SurvBoard genes (union):", len(survboard_genes))
print("SurvBoard genes (intersection):", len(survboard_genes_intersection))

ARCHS4 genes: 67186
DepMap genes: 17742
DisSignAtlas genes: 18881
GDSC genes: 18916
GTEx genes: 74628
LINCS genes: 946
TCGA genes: 22777
SurvBoard genes (union): 16522
SurvBoard genes (intersection): 13575


In [19]:
# LINCS is much smaller than the others, so we take the intersection of all datasets except LINCS to define our core gene list, and then see how many of those are in LINCS.
gene_list_ensemble_id = set(
    archs4_genes
    & depmap_genes
    & disignatlas_genes
    & gdsc_genes
    & gtex_genes
    & survboard_genes_intersection
    & tcga_genes
)


In [20]:
len(gene_list_ensemble_id)

13004

In [21]:
# Check whether all LINCS genes are in the core gene list
len(gene_list_ensemble_id & lincs_genes)

910

In [22]:
genes = sorted(list(gene_list_ensemble_id))

with open("/cluster/work/boeva/eheiss/datasets/gene_list.txt", "w") as f:
    for g in genes:
        f.write(g + "\n")

In [5]:
# query GENExCELL to see whether full gene_list is available there (internet access required, so run this cell separately if needed)

from pathlib import Path
import cellxgene_census

gene_list_path = Path("/Users/enricoheiss/Thesis/eheiss/scbFM/data/gene_list.txt")
census_version = "2025-11-08"
organism_key = "homo_sapiens"

with open(gene_list_path) as f:
    gene_list = [line.strip() for line in f if line.strip()]

with cellxgene_census.open_soma(census_version=census_version) as census:
    var = census["census_data"][organism_key].ms["RNA"].var.read(
        column_names=["soma_joinid", "feature_id", "feature_name"]
    ).concat().to_pandas()

available_feature_ids = set(var["feature_id"].astype(str))
available_feature_names = set(var["feature_name"].astype(str))

missing_by_ensg = sorted(set(gene_list) - available_feature_ids)
available_by_ensg = sorted(set(gene_list) & available_feature_ids)

print(f"Gene list size: {len(gene_list)}")
print(f"Available in CELLxGENE by Ensembl feature_id: {len(available_by_ensg)}")
print(f"Missing in CELLxGENE by Ensembl feature_id: {len(missing_by_ensg)}")

if missing_by_ensg:
    print("Missing genes:")
    for gene in missing_by_ensg:
        print(gene)
else:
    print("All genes from gene_list are available in CELLxGENE.")

Gene list size: 13004
Available in CELLxGENE by Ensembl feature_id: 13004
Missing in CELLxGENE by Ensembl feature_id: 0
All genes from gene_list are available in CELLxGENE.
